# GPT API 테스트 노트북

OpenAI Python SDK(v2.x)를 사용한 GPT API 호출 예제입니다.

- 공식 문서 기준으로 현재 표준 API는 **Responses API** (`client.responses.create`)입니다.
- API 키는 프로젝트 루트의 `.env` 파일에 `OPENAI_API_KEY`로 저장합니다.

> 실행 전 준비: `.env` 파일을 열어 `OPENAI_API_KEY=` 뒤에 발급받은 키를 붙여넣으세요.

## 1. 환경 설정 — API 키 로드 및 클라이언트 초기화

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
assert api_key and not api_key.startswith("여기에"), ".env 파일에 OPENAI_API_KEY를 설정해주세요!"

# api_key 인자를 생략해도 OPENAI_API_KEY 환경변수를 자동으로 읽습니다
client = OpenAI()

# 사용할 모델 — 필요에 따라 변경 (예: gpt-5.5, gpt-5-mini 등)
MODEL = "gpt-5-nano"

print("클라이언트 초기화 완료 ✅")

클라이언트 초기화 완료 ✅


## 2. 기본 텍스트 생성 (Responses API)

`client.responses.create()`가 현재 공식 문서에서 권장하는 텍스트 생성 방식입니다.

In [11]:
response = client.responses.create(
    model=MODEL,
    input="서울, 도쿄, 파리 중 지금 가장 따뜻한 도시를 찾고, 그 도시 통화로 500달러를 바꾸면 얼마인지 알려줘",
        reasoning={
        "effort": "high",     # none/minimal/low/medium/high/xhigh/max
        "summary": "auto",    # auto/concise/detailed
    },
)

response.output  # 모델의 응답 출력

[ResponseReasoningItem(id='rs_07a1e62d92c150b7006a61b600d3e0819ab7d05a188587adaf', summary=[Summary(text="**Considering weather comparisons**\n\nI'm looking at the weather in different cities: Seoul might be around 32°F, Tokyo around 30°F, and Paris at 25°F. I think these comparisons might help someone who's curious about global temperatures, but I need to be careful about how I present this information. It's important to clarify whether these are Celsius or Fahrenheit, as it can significantly change perceptions! I want to make sure I'm communicating clearly.", type='summary_text')], type='reasoning', content=[], encrypted_content='gAAAAABqYbYLafmYy_AvLVjYN_Mja9PTtDMnqU1jlWgLwwYjwqhkZyaKfh2BwOVOeOUhXeafuAJuWI5Fcq2fZODKK6grE5vZh7qsrRmCSKA24RzUTK91yMZ4dB9D5HeybA-gf-D1R9aCWGt1TkHdT6RNKHAlv4IzB71T4CFAo6O9IommhJJPRiiW-zMqFI9ClrnOWDUV9Sqf4tomkhuxGcQlXF5nAxKOmJee0urkvoL-lgGHG2ArKugZ-gCnKLTGe3IrERYb_1RcwypLYdASac5jbUkGl8li1OgkSqtYnfBsugt0uQ3fveUHKC4AyZ6VqtfXqIRe8WYnou96OBMvkx5KrwleIcq5LOLVZzO9

## 3. 시스템 지시(instructions) 추가

`instructions` 파라미터로 모델의 역할·말투를 지정할 수 있습니다.

In [9]:
response = client.responses.create(
    model=MODEL,
    instructions="당신은 친절한 코딩 튜터입니다. 초보자 눈높이에 맞춰 한국어로 설명하세요.",
    input="파이썬 데코레이터가 뭐야?",
    reasoning={"summary": "auto"},
)

print(response.output_text)

파이썬 **데코레이터(decorator)**는 쉽게 말해, **기존 함수의 코드를 직접 바꾸지 않고 기능을 추가하거나 수정하는 문법**입니다.

예를 들어 어떤 함수가 실행되기 전/후에 로그를 찍거나, 실행 시간을 재거나, 권한 검사를 할 때 자주 씁니다.

---

## 1. 아주 간단한 예시

```python
def hello():
    print("안녕하세요!")
```

이 함수가 실행될 때 앞뒤로 메시지를 추가하고 싶다고 해봅시다.

```python
def decorator(func):
    def wrapper():
        print("함수 실행 전")
        func()
        print("함수 실행 후")
    return wrapper
```

이제 `hello` 함수에 데코레이터를 적용할 수 있습니다.

```python
hello = decorator(hello)

hello()
```

결과:

```text
함수 실행 전
안녕하세요!
함수 실행 후
```

즉, `hello()`를 실행했지만 실제로는 `wrapper()`가 실행되면서 기존 `hello()` 앞뒤에 기능이 추가된 것입니다.

---

## 2. `@` 문법 사용하기

파이썬에서는 위 코드를 더 간단하게 쓸 수 있습니다.

```python
def decorator(func):
    def wrapper():
        print("함수 실행 전")
        func()
        print("함수 실행 후")
    return wrapper


@decorator
def hello():
    print("안녕하세요!")


hello()
```

결과는 똑같습니다.

```text
함수 실행 전
안녕하세요!
함수 실행 후
```

여기서:

```python
@decorator
def hello():
    ...
```

는 사실상 아래 코드와 같습니다.

```python
hello = decorator(hello)
``

## 4. 스트리밍 출력

`stream=True`를 주면 토큰이 생성되는 대로 실시간으로 받아볼 수 있습니다.

In [ ]:
stream = client.responses.create(
    model=MODEL,
    input="머신러닝과 딥러닝의 차이를 세 문장으로 설명해줘.",
    stream=True,
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

## 5. 대화 이어가기 (멀티턴)

`previous_response_id`로 이전 응답에 이어서 대화할 수 있습니다. 직접 대화 기록을 관리할 필요가 없습니다.

In [ ]:
first = client.responses.create(
    model=MODEL,
    input="태양계에서 가장 큰 행성은 뭐야?",
)
print("[1턴]", first.output_text)

second = client.responses.create(
    model=MODEL,
    previous_response_id=first.id,
    input="그 행성의 위성은 몇 개야?",
)
print("[2턴]", second.output_text)

## 참고: Chat Completions API (이전 표준)

기존 방식인 `chat.completions`도 계속 지원됩니다. 레거시 코드 호환이 필요할 때 사용하세요.

In [ ]:
completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "간결하게 답변하세요."},
        {"role": "user", "content": "REST API가 뭔지 한 문장으로 설명해줘."},
    ],
)

print(completion.choices[0].message.content)